# MuScriptor — Audio to Multi-Instrument Sheet Music
Official model: https://github.com/muscriptor/muscriptor

T4 GPU Colab notebook. Outputs multi-instrument MIDI, MusicXML, and (with MuseScore 4+) sheet-music PDFs. The published model weights are **CC BY-NC 4.0**.

Before running, accept the license on the selected Hugging Face model page: [small](https://huggingface.co/MuScriptor/muscriptor-small) / [medium](https://huggingface.co/MuScriptor/muscriptor-medium) / [large](https://huggingface.co/MuScriptor/muscriptor-large).

## 1. Install MuScriptor

In [ ]:
%pip -q install uv huggingface_hub
import subprocess
subprocess.run(["uv", "python", "install", "3.12"], check=True)
subprocess.run(["uv", "venv", "/content/muscriptor_env", "--python", "3.12"], check=True)
subprocess.run(["uv", "pip", "install", "--python", "/content/muscriptor_env/bin/python", "git+https://github.com/muscriptor/muscriptor.git"], check=True)
muscriptor_cli = "/content/muscriptor_env/bin/muscriptor"

## 2. Hugging Face login
Use a Hugging Face read token with access to the accepted model license.

In [ ]:
from huggingface_hub import login
login()

## 3. Upload original song

In [ ]:
from google.colab import files
from pathlib import Path
import subprocess

uploads = files.upload()
input_dir = Path("/content/muscriptor_input")
input_dir.mkdir(parents=True, exist_ok=True)
audio_paths = []
for filename, data in uploads.items():
    source = input_dir / filename
    source.write_bytes(data)
    audio_paths.append(source)
    print(source)

## 4. Install MuseScore 4+ for sheet PDF export
MuScriptor requires MuseScore 4 or newer for `--format sheets`. This cell fetches a Linux x86-64 AppImage from the official MuseScore GitHub releases and extracts it for headless Colab execution.

In [ ]:
import json
import os
import subprocess
import urllib.request
from pathlib import Path

releases_api = "https://api.github.com/repos/musescore/MuseScore/releases/latest"
req = urllib.request.Request(releases_api, headers={"User-Agent": "colab-muscriptor"})
with urllib.request.urlopen(req) as response:
    release = json.load(response)
assets = release["assets"]
matches = [a for a in assets if a["name"].endswith(".AppImage") and "x86_64" in a["name"]]
if not matches:
    raise RuntimeError("No x86_64 MuseScore AppImage found in the official latest release")
asset = matches[0]
appimage_path = Path("/content/MuseScore4.AppImage")
urllib.request.urlretrieve(asset["browser_download_url"], appimage_path)
appimage_path.chmod(0o755)
subprocess.run([str(appimage_path), "--appimage-extract"], check=True, cwd="/content", stdout=subprocess.DEVNULL)
musescore_executable = Path("/content/squashfs-root/AppRun")
os.environ["MUSCRIPTOR_MUSESCORE"] = str(musescore_executable)
print("MuseScore:", musescore_executable)

## 5. Transcribe original audio into a full score
Model `medium` is MuScriptor's official default; select `small` if desired. This invokes the documented sheet export CLI.

In [ ]:
#@title MuScriptor 채보 — 실시간 로그
MODEL = "medium" #@param ["small", "medium", "large"]

from pathlib import Path
import os
import subprocess

score_root = Path("/content/muscriptor_scores")
score_root.mkdir(parents=True, exist_ok=True)

env = dict(os.environ)
env["QT_QPA_PLATFORM"] = "offscreen"
env["MUSCRIPTOR_MUSESCORE"] = str(musescore_executable)
env["PYTHONUNBUFFERED"] = "1"

for audio in audio_paths:
    destination = score_root / audio.stem
    command = [
        muscriptor_cli, "transcribe", str(audio),
        "--model", MODEL,
        "--format", "sheets",
        "--output", str(destination)
    ]
    print(f"Processing: {audio.name}", flush=True)
    with subprocess.Popen(
        command,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(f"MuScriptor failed (exit code {returncode})")
    print(f"Completed: {destination}", flush=True)

## 6. List generated scores and download
Includes whole-score PDF, per-instrument PDFs, MusicXML, and MIDI.

In [ ]:
from pathlib import Path
from google.colab import files

all_scores = sorted(p for p in score_root.rglob("*") if p.is_file())
for i, path in enumerate(all_scores):
    print(f"{i:02d}  {path.relative_to(score_root)}")

DOWNLOAD_INDEX = 0 #@param {type:"integer"}
files.download(str(all_scores[DOWNLOAD_INDEX]))

## 7. SongFormer — Verse / Pre-Chorus / Chorus boundaries
Runs the same SongFormer inference used in the separate melody notebook. Requires GPU memory for SongFormer in addition to the MuScriptor transcription.

In [ ]:
%pip -q install uv
import subprocess
from pathlib import Path
subprocess.run(["uv", "python", "install", "3.10"], check=True)
subprocess.run(["uv", "venv", "/content/songformer_env", "--python", "3.10"], check=True)
songformer_python = "/content/songformer_env/bin/python"
songformer_root = Path("/content/SongFormer")
if not songformer_root.exists():
    subprocess.run(["git", "clone", "--recursive", "https://github.com/ASLP-lab/SongFormer.git", str(songformer_root)], check=True)
subprocess.run(["uv", "pip", "install", "--python", songformer_python, "-r", str(songformer_root / "requirements.txt")], check=True)
subprocess.run(["uv", "pip", "install", "--python", songformer_python, "setuptools==80.9.0"], check=True)

In [ ]:
from pathlib import Path
import subprocess, json, os

songformer_input = Path("/content/music_input")
songformer_input.mkdir(parents=True, exist_ok=True)
for audio in audio_paths:
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(audio),
                    "-ar", "44100", "-ac", "2",
                    str(songformer_input / f"{audio.stem}.wav")], check=True)

songformer_script = Path("/content/run_songformer.py")
songformer_script.write_text("""
import os, sys, json
from pathlib import Path
from huggingface_hub import snapshot_download
from transformers import AutoModel
local_dir = snapshot_download(repo_id="ASLP-lab/SongFormer", repo_type="model",
                              ignore_patterns=["SongFormer.pt", "SongFormer.safetensors"])
sys.path.append(local_dir)
os.environ["SONGFORMER_LOCAL_DIR"] = local_dir
model = AutoModel.from_pretrained(local_dir, trust_remote_code=True, low_cpu_mem_usage=False)
model.to("cuda:0")
model.eval()
results = {}
for wav in sorted(Path("/content/music_input").glob("*.wav")):
    results[wav.stem] = model(str(wav))
Path("/content/songformer_sections.json").write_text(json.dumps(results, ensure_ascii=False, indent=2))
""")
subprocess.run([songformer_python, str(songformer_script)], check=True,
               env={**os.environ, "MPLBACKEND":"Agg"})
sections_by_track = json.loads(Path("/content/songformer_sections.json").read_text())
for track, sections in sections_by_track.items():
    print(track)
    for section in sections:
        print(section)

## 8. Extract MIDI melody and estimated chord sequence per detected section
Chord names are estimates computed from concurrent pitched notes in the transcribed MIDI, not labels provided by MuScriptor. The original MIDI pitches and tempo are preserved.

In [ ]:
%pip -q install pretty_midi
from pathlib import Path
from collections import Counter
import pretty_midi
import json
import math

PC_NAMES = ["C", "C#", "D", "Eb", "E", "F", "F#", "G", "Ab", "A", "Bb", "B"]
CHORD_TEMPLATES = {
    "": (0,4,7), "m": (0,3,7), "dim": (0,3,6), "aug": (0,4,8),
    "7": (0,4,7,10), "maj7": (0,4,7,11), "m7": (0,3,7,10)
}
TARGET = {"verse","pre-chorus","chorus"}

def estimate_chord(notes, start, end):
    weight = [0.0] * 12
    for note in notes:
        duration = max(0.0, min(end,note.end) - max(start,note.start))
        weight[note.pitch % 12] += duration
    if not any(weight):
        return "N"
    best = None
    for root in range(12):
        for suffix, intervals in CHORD_TEMPLATES.items():
            pcs = {(root + x) % 12 for x in intervals}
            score = sum(weight[p] for p in pcs) - 0.65 * sum(weight[p] for p in range(12) if p not in pcs)
            if best is None or score > best[0]:
                best = (score, PC_NAMES[root] + suffix)
    return best[1]

def melody_notes(voice, start, end):
    return [
        pretty_midi.note_number_to_name(n.pitch)
        for n in sorted(voice.notes, key=lambda n:n.start)
        if start <= n.start < end
    ]

extracted = {}
for audio in audio_paths:
    score_midi = score_root / audio.stem / "score.mid"
    midi = pretty_midi.PrettyMIDI(str(score_midi))
    voice_tracks = [inst for inst in midi.instruments
                    if not inst.is_drum and ("voice" in inst.name.lower() or "women" in inst.name.lower()
                                            or "vocal" in inst.name.lower())]
    accompaniment = [n for inst in midi.instruments if not inst.is_drum
                     and inst not in voice_tracks for n in inst.notes]
    sections = sections_by_track[audio.stem]
    track_results = []
    for idx, sec in enumerate(sections, 1):
        label = sec["label"].strip().lower().replace("_","-").replace(" ","-")
        if label not in TARGET:
            continue
        start, end = float(sec["start"]),float(sec["end"])
        notes = [note for voice in voice_tracks for note in melody_notes(voice,start,end)]
        # Fixed 2-second windows preserve the order of harmonic changes.
        chord_windows = []
        window_start = start
        while window_start < end:
            window_end = min(end, window_start+2.0)
            chord_windows.append({"start":round(window_start,2),
                                  "end":round(window_end,2),
                                  "chord":estimate_chord(accompaniment,window_start,window_end)})
            window_start = window_end
        track_results.append({"section":label,"start":start,"end":end,
                              "melody":notes,"chords":chord_windows})
    extracted[audio.stem] = {"tempo_changes":midi.get_tempo_changes()[1].tolist(),
                            "sections":track_results}

out_path = Path("/content/muscriptor_section_analysis.json")
out_path.write_text(json.dumps(extracted, ensure_ascii=False, indent=2))
for title, data in extracted.items():
    print(title, "MIDI tempo(s):", data["tempo_changes"])
    for sec in data["sections"]:
        print(f'[{sec["section"]}] {sec["start"]:.2f}–{sec["end"]:.2f}s')
        print("Melody:", " ".join(sec["melody"]))
        print("Chords (estimated):", " | ".join(x["chord"] for x in sec["chords"]))
        print()
print("Saved:", out_path)